In [ ]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import requests
import re

MONSTERS_SELECTED = ['aarakocra', 'basilisk', 'gorgon', 'mind-flayer', 'yuan-ti-abomination']
url = "https://www.aidedd.org/dnd-filters/monsters.php"

In [ ]:
def get_soup(url: str) -> BeautifulSoup:
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    return soup

In [ ]:
soup = get_soup(url)

In [ ]:
tabla = soup.find("table")
print(len(tabla))
rows = tabla.find_all("tr")


for row in rows[1:]:
    celdas = row.find_all("td")
print(celdas[8].prettify()[:3000])



In [ ]:
def generate_database(soup: BeautifulSoup) -> pd.DataFrame:
    tabla = soup.find("table", class_ ="liste")
    rows = tabla.find_all("tr")
    filas = []
    for row in rows[1:]:
        celdas = row.find_all("td")
        criatura = celdas[1].get_text(strip = True)
        typo = celdas[3].get_text(strip = True)
        alignment = celdas[8].get_text(strip = True)

        filas.append({
            "Creature" : criatura,
            "Type" : typo,
            "Alignment" : alignment
        })

    df = pd.DataFrame(filas)
    return df




In [ ]:
df = generate_database(soup)

In [ ]:
def make_uniform_name(monsters_df_1c: pd.DataFrame) -> pd.DataFrame:
    df["Creature"] = df["Creature"].str.replace(" ", "-", regex  =False)
    df["Creature"] = df["Creature"].str.lower()
    return df

In [ ]:
df=make_uniform_name(df)

In [ ]:
def get_api_database(monsters_selected: list[str]) -> pd.DataFrame:
    base_url = "https://www.dnd5eapi.co"
    filas =[]
    for monster in monsters_selected:
        url = base_url + "/api/2014/monsters/" + monster
        response = requests.get(url)

        if response.status_code == 200:
            data = response.json()
            strength = data.get("strength", np.nan)
            wisdom = data.get("wisdom", np.nan)
            charisma = data.get("charisma", np.nan)

        else:
            strength = np.nan
            wisdom = np.nan
            charisma = np.nan
        
        filas.append({
            "Creature" : monster,
            "Strength" : strength,
            "Wisdom" : wisdom,
            "Charisma" : charisma
        })


    df = pd.DataFrame(filas)
    return df

In [ ]:
df_api = get_api_database(MONSTERS_SELECTED)

In [ ]:
monsters_df_3_web = pd.read_csv("monsters_3_web.csv", sep = ";")
monsters_df_3_api = pd.read_csv("monsters_3_api.csv", sep=";")

In [ ]:
def join_sources(monsters_df_3_web: pd.DataFrame, 
                 monsters_df_3_api: pd.DataFrame) -> pd.DataFrame:
    df_unido = pd.merge(monsters_df_3_web, monsters_df_3_api, on="Creature", how="left")
    df_unido = df_unido[['Creature', 'Type', 'Second Type', 'Alignment', 'Strength', 'Wisdom', 'Charisma']]
    return df_unido

In [ ]:
df_unido = join_sources(monsters_df_3_web,monsters_df_3_api)

In [ ]:
def impute_missing(monsters_df_3b: pd.DataFrame) -> pd.DataFrame:
    mediana = monsters_df_3b.groupby("Alignment")["Strength"].transform("median")
    monsters_df_3b["Strength"] = monsters_df_3b["Strength"].fillna(mediana)
    return monsters_df_3b

In [ ]:
df = impute_missing(df_unido)

In [ ]:
def delete_outliers(monsters_df_3c: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    q1 = monsters_df_3c["Strength"].quantile(0.25)
    q3 = monsters_df_3c["Strength"].quantile(0.75)
    iqr = q3- q1

    lim_inf = q1 - 2.5*iqr
    lim_sup = q3 + 2.5*iqr

    outliers = monsters_df_3c[(monsters_df_3c["Strength"] < lim_inf) | (monsters_df_3c["Strength"] > lim_sup)]
    no_outliers = monsters_df_3c[(monsters_df_3c["Strength"] >= lim_inf) & (monsters_df_3c["Strength"] <= lim_sup)]
    return outliers, no_outliers

In [ ]:
delete_outliers(df)